# 面试问题：DoRA 如何把权重幅值与方向解耦，它和 LoRA、merge、归一化梯度及发布制品是什么关系？

**一句话回答。** LoRA 冻结预训练权重 $W_0$，只学习低秩增量 $\Delta W=sBA$；DoRA 仍用低秩矩阵产生方向候选 $V=W_0+sBA$，但进一步把最终权重写成

$$W'=m\odot\frac{V}{\lVert V\rVert},$$

其中 $m$ 单独学习每个权重向量的幅值，归一化后的 $V$ 负责方向。这样做的动机是让参数高效微调更接近全量微调中“幅值变化”和“方向变化”呈现的不同学习行为，而不是把二者都挤在同一个低秩增量中。

本 Notebook 只使用 PyTorch 基础张量与 `nn.Module`，手写 LoRA、DoRA、稳定归一化、反向传播、手工梯度更新和 merge。线性层权重存储为 `[输出维, 输入维]`，所以代码按每个输出行求范数；若论文公式把权重向量写成列，这只是转置约定不同。主要参考：[DoRA: Weight-Decomposed Low-Rank Adaptation](https://arxiv.org/abs/2402.09353)。

In [ ]:
import torch  # 导入 PyTorch 张量、自动微分和基础神经网络能力。
from torch import nn  # 导入模块与可训练参数基类以手写线性适配器。
torch.manual_seed(239)  # 固定随机种子，让所有初始化和断言可以重复执行。
input_size = 6  # 设置微型线性层的输入特征维度。
output_size = 4  # 设置微型线性层的输出特征维度。
rank = 2  # 设置低秩适配器的秩以展示参数节省。
epsilon = 1e-6  # 设置归一化分母下界以避免零向量除零。
assert torch.__version__  # 验证当前环境已经成功载入 PyTorch。
assert 0 < rank < min(input_size, output_size)  # 验证低秩维度确实小于原权重维度。
assert epsilon > 0.0  # 验证归一化保护常数为正数。
assert torch.initial_seed() == 239  # 验证随机种子与本实验编号一致。

## 1. 先把 LoRA 写清楚：它学习的是低秩增量

对于线性层 $y=xW_0^\top+b$，LoRA 将有效权重改成 $W_0+sBA$，其中 $A\in\mathbb{R}^{r\times d_{in}}$、$B\in\mathbb{R}^{d_{out}\times r}$、$s=\alpha/r$。训练时冻结 $W_0$，只更新 $A,B$；常见初始化让 $B=0$，因此适配开始时模型行为与基座完全一致。

LoRA 的重要工程优势是可插拔与可合并：训练制品可以只保存小矩阵，部署时也可以预先计算 $W_0+sBA$。但这个增量同时改变向量长度和方向，它没有显式告诉优化器“这部分参数只负责尺度，那部分只负责方向”。下一节正是在这个接口上加入 DoRA 分解。

In [ ]:
class LoRALinear(nn.Module):  # 定义只依赖基础矩阵乘法的 LoRA 线性层。
    def __init__(self, base_weight, base_bias, adapter_rank, alpha):  # 接收冻结权重、偏置、秩和缩放超参数。
        super().__init__()  # 初始化模块基类以登记参数与缓冲区。
        self.register_buffer("base_weight", base_weight.detach().clone())  # 把预训练权重保存为不参与梯度的缓冲区。
        self.register_buffer("base_bias", base_bias.detach().clone())  # 把预训练偏置保存为不参与梯度的缓冲区。
        self.adapter_a = nn.Parameter(torch.randn(adapter_rank, base_weight.shape[1]) * 0.02)  # 随机初始化负责输入投影的低秩矩阵 A。
        self.adapter_b = nn.Parameter(torch.zeros(base_weight.shape[0], adapter_rank))  # 用零初始化矩阵 B 保证起始增量为零。
        self.scale = alpha / adapter_rank  # 按 LoRA 约定计算与秩相关的增量缩放系数。
    def delta_weight(self):  # 计算当前低秩适配器产生的完整形状权重增量。
        return self.scale * (self.adapter_b @ self.adapter_a)  # 用两个窄矩阵乘积构造低秩增量。
    def effective_weight(self):  # 计算推理时真正参与线性变换的有效权重。
        return self.base_weight + self.delta_weight()  # 把冻结基座与可训练低秩增量相加。
    def forward(self, inputs):  # 使用有效权重完成线性层前向传播。
        return inputs @ self.effective_weight().transpose(0, 1) + self.base_bias  # 手写矩阵乘法并加上冻结偏置。
base_weight = torch.randn(output_size, input_size) * 0.2  # 构造一个微型预训练权重矩阵。
base_bias = torch.randn(output_size) * 0.05  # 构造一个微型预训练偏置向量。
sample_inputs = torch.randn(5, input_size)  # 构造五条用于接口验证的输入样本。
lora_layer = LoRALinear(base_weight, base_bias, rank, alpha=4.0)  # 创建秩为二的 LoRA 线性层。
lora_outputs = lora_layer(sample_inputs)  # 执行初始化状态下的 LoRA 前向传播。
base_outputs = sample_inputs @ base_weight.transpose(0, 1) + base_bias  # 直接计算冻结基座的参考输出。
assert lora_outputs.shape == torch.Size([5, output_size])  # 验证 LoRA 前向输出形状正确。
assert torch.count_nonzero(lora_layer.delta_weight()).item() == 0  # 验证零初始化 B 使初始增量严格为零。
assert torch.allclose(lora_outputs, base_outputs)  # 验证适配器初始化不会改变基座行为。
assert lora_layer.base_weight.requires_grad is False  # 验证预训练权重没有加入梯度更新。

## 2. DoRA 的核心：把每个权重向量拆成 magnitude 与 direction

任意非零向量都可以写成“长度乘单位方向”。在本代码的 `[输出维, 输入维]` 布局中，第 $i$ 个输出神经元对应权重行 $w_i$，于是初始化为 $m_i=\lVert w_i\rVert_2$、$u_i=w_i/m_i$，并有 $w_i=m_i u_i$。DoRA 把 $m_i$ 变成独立可训练参数，而方向候选由冻结权重与低秩增量共同构造。

实现时必须处理近零向量：分母用 FP32 累加并限制到 `epsilon`，否则半精度平方和、极小范数或异常 checkpoint 都可能产生 `NaN`。零向量归一化后仍保持零，而不是伪造一个任意方向；生产系统应额外报告这种退化行并决定拒绝、重置还是继续训练。

In [ ]:
def stable_row_norm(weight, minimum=1e-6):  # 按输出行计算稳定的二范数并保留广播维度。
    working_weight = weight.float()  # 先转成 FP32 计算平方和以减小半精度误差。
    squared_sum = torch.sum(working_weight * working_weight, dim=1, keepdim=True)  # 汇总每个输出行的元素平方。
    protected_norm = torch.sqrt(squared_sum).clamp_min(minimum)  # 开方并用下界保护可能的零范数。
    return protected_norm.to(weight.dtype)  # 把范数转换回输入权重的数据类型。
def stable_row_normalize(weight, minimum=1e-6):  # 把每个非零输出行归一化为单位方向。
    return weight / stable_row_norm(weight, minimum)  # 用带保护的行范数执行逐行广播除法。
initial_magnitude = stable_row_norm(base_weight, epsilon)  # 从预训练权重提取每个输出行的初始幅值。
initial_direction = stable_row_normalize(base_weight, epsilon)  # 从预训练权重提取归一化方向。
reconstructed_weight = initial_magnitude * initial_direction  # 将幅值和方向相乘重建原权重。
zero_probe = torch.zeros(2, input_size)  # 构造全零权重以检查分母保护逻辑。
normalized_zero_probe = stable_row_normalize(zero_probe, epsilon)  # 对退化零向量执行稳定归一化。
assert initial_magnitude.shape == torch.Size([output_size, 1])  # 验证每个输出行只有一个独立幅值。
assert torch.allclose(stable_row_norm(initial_direction), torch.ones(output_size, 1))  # 验证非零方向行已经归一化为单位长度。
assert torch.allclose(reconstructed_weight, base_weight, atol=1e-6)  # 验证幅值方向分解可以无损重建基座权重。
assert torch.isfinite(normalized_zero_probe).all()  # 验证全零输入不会因除零产生非数值。
assert torch.count_nonzero(normalized_zero_probe).item() == 0  # 验证退化零方向不会被保护常数凭空改变。

## 3. 从零实现 DoRA 线性层

DoRA 与 LoRA 并不是二选一：低秩矩阵仍负责高效地产生方向候选，新增的 `magnitude` 参数负责每个输出向量的长度。初始化时 `B=0` 且 `magnitude=||W0||`，所以有效权重严格回到 $W_0$。前向时先算 $V=W_0+sBA$，再归一化并乘 $m$。

这里将基座权重和偏置登记为 buffer，而不是可训练 Parameter；这样优化器只会看到 `A`、`B` 和 `magnitude`。真实 Transformer 通常只给部分投影层加 DoRA，还要明确是否适配 bias、张量并行如何切分 magnitude、共享权重是否允许独立分解，这些都应进入训练清单而不能依赖默认行为。

In [ ]:
class DoRALinear(nn.Module):  # 定义具有独立幅值和低秩方向更新的 DoRA 线性层。
    def __init__(self, base_weight, base_bias, adapter_rank, alpha, minimum=1e-6):  # 接收基座张量与 DoRA 超参数。
        super().__init__()  # 初始化模块基类以登记可训练参数和缓冲区。
        self.register_buffer("base_weight", base_weight.detach().clone())  # 保存冻结且可随制品迁移的预训练权重。
        self.register_buffer("base_bias", base_bias.detach().clone())  # 保存冻结的线性层偏置。
        self.adapter_a = nn.Parameter(torch.randn(adapter_rank, base_weight.shape[1]) * 0.02)  # 初始化低秩方向矩阵 A。
        self.adapter_b = nn.Parameter(torch.zeros(base_weight.shape[0], adapter_rank))  # 零初始化方向矩阵 B 以保持初始等价。
        self.magnitude = nn.Parameter(stable_row_norm(base_weight, minimum).detach().clone())  # 把基座行范数注册成独立可训练幅值。
        self.scale = alpha / adapter_rank  # 计算低秩方向更新的缩放系数。
        self.minimum = minimum  # 保存归一化分母下界供每次前向使用。
    def directional_weight(self):  # 构造尚未归一化的方向候选矩阵 V。
        return self.base_weight + self.scale * (self.adapter_b @ self.adapter_a)  # 将低秩方向增量加到冻结基座上。
    def effective_weight(self):  # 由独立幅值和单位方向构造最终有效权重。
        unit_direction = stable_row_normalize(self.directional_weight(), self.minimum)  # 对每个输出行执行稳定归一化。
        return self.magnitude * unit_direction  # 用独立可训练幅值缩放每个单位方向。
    def forward(self, inputs):  # 使用 DoRA 有效权重完成线性变换。
        return inputs @ self.effective_weight().transpose(0, 1) + self.base_bias  # 手写线性前向并加上冻结偏置。
dora_layer = DoRALinear(base_weight, base_bias, rank, alpha=4.0, minimum=epsilon)  # 创建与前面基座对应的 DoRA 层。
dora_initial_outputs = dora_layer(sample_inputs)  # 执行 DoRA 初始化状态的前向传播。
trainable_names = {name for name, parameter in dora_layer.named_parameters() if parameter.requires_grad}  # 收集真正参与优化的参数名称。
assert torch.allclose(dora_layer.effective_weight(), base_weight, atol=1e-6)  # 验证 DoRA 初始化有效权重等于预训练权重。
assert torch.allclose(dora_initial_outputs, base_outputs, atol=1e-6)  # 验证 DoRA 初始化前向行为与基座一致。
assert trainable_names == {"adapter_a", "adapter_b", "magnitude"}  # 验证只有低秩矩阵和幅值会被训练。
assert dora_layer.magnitude.shape == torch.Size([output_size, 1])  # 验证幅值参数可以逐输出行广播。
assert torch.count_nonzero(dora_layer.adapter_b).item() == 0  # 验证零初始化仍保留无扰动起点。

## 4. 为什么解耦不是换一种写法：径向缩放会被方向归一化消掉

如果把方向候选的某一行乘以正数 $c$，归一化后仍是同一个单位向量；因此纯径向变化不会偷偷改变方向。相反，修改 $m_i$ 会直接改变对应有效权重行的长度，而不会旋转它。这让优化器拥有一条显式的“幅值通道”。

需要注意的是，`magnitude` 通常不应被解释成模型输出的置信度，它只是参数空间中的行范数。幅值增大可能强化某个投影，也可能被后续归一化层、残差或激活抵消。面试回答应把参数几何直觉与端到端任务性能分开，并通过消融、权重范数轨迹和下游指标验证。

In [ ]:
direction_candidate = dora_layer.directional_weight().detach().clone()  # 复制当前尚未归一化的方向候选用于几何实验。
positive_row_scales = torch.tensor([[0.5], [2.0], [3.0], [1.5]])  # 为四个输出行指定不同的正径向缩放。
rescaled_candidate = direction_candidate * positive_row_scales  # 只改变候选行长度而不改变其朝向。
original_unit_direction = stable_row_normalize(direction_candidate, epsilon)  # 计算原候选的单位方向。
rescaled_unit_direction = stable_row_normalize(rescaled_candidate, epsilon)  # 计算径向缩放后的单位方向。
larger_magnitude = dora_layer.magnitude.detach() * 1.5  # 构造整体放大一点五倍的独立幅值。
original_weight_without_bias = dora_layer.magnitude.detach() * original_unit_direction  # 用原幅值构造无偏置权重。
larger_weight_without_bias = larger_magnitude * original_unit_direction  # 用更大幅值和相同方向构造权重。
original_linear_outputs = sample_inputs @ original_weight_without_bias.transpose(0, 1)  # 计算不含偏置的原幅值输出。
larger_linear_outputs = sample_inputs @ larger_weight_without_bias.transpose(0, 1)  # 计算不含偏置的放大幅值输出。
assert torch.allclose(original_unit_direction, rescaled_unit_direction, atol=1e-6)  # 验证正径向缩放会被归一化完全消除。
assert torch.allclose(stable_row_norm(original_weight_without_bias), dora_layer.magnitude.detach(), atol=1e-6)  # 验证有效权重行范数等于显式幅值。
assert not torch.allclose(original_weight_without_bias, larger_weight_without_bias)  # 验证修改幅值确实改变最终权重。
assert torch.allclose(larger_linear_outputs, original_linear_outputs * 1.5, atol=1e-6)  # 验证无偏置线性输出随整体幅值同比例变化。

## 5. 梯度怎样流动：零初始化时 A 暂时没有梯度并不是故障

当 $B=0$ 时，$BA=0$，损失对 $B$ 通常有梯度，但对 $A$ 的链式梯度包含 $B^\top$，所以第一步 `A.grad` 为零是预期现象；更新 $B$ 后，下一步梯度即可流入 $A$。`magnitude` 从第一步就能直接接收梯度。这个现象同时存在于常见的 LoRA 零初始化方案中，排查训练时不能把首步 A 梯度为零误报成断图。

方向归一化的梯度会去掉径向分量，使方向参数主要沿单位球面的切向更新；幅值参数则承担径向变化。本节使用一个已知教师权重构造回归目标，不调用优化器封装，而是显式执行 `parameter -= learning_rate * gradient`，从而看到每组参数的更新次序。

In [ ]:
student = DoRALinear(base_weight, base_bias, rank, alpha=2.0, minimum=epsilon)  # 创建新的 DoRA 学生层用于梯度实验。
frozen_weight_snapshot = student.base_weight.detach().clone()  # 保存训练前基座副本以验证冻结语义。
teacher_update_a = torch.tensor([[0.20, -0.10, 0.05, 0.00, 0.10, -0.05], [-0.10, 0.15, 0.00, 0.10, -0.05, 0.20]])  # 构造教师低秩方向矩阵 A。
teacher_update_b = torch.tensor([[0.10, -0.05], [0.00, 0.15], [-0.10, 0.10], [0.05, 0.05]])  # 构造教师低秩方向矩阵 B。
teacher_direction = stable_row_normalize(base_weight + teacher_update_b @ teacher_update_a, epsilon)  # 构造教师单位方向。
teacher_magnitude = stable_row_norm(base_weight, epsilon) * torch.tensor([[1.20], [0.85], [1.10], [0.90]])  # 为教师设置独立的逐行目标幅值。
teacher_weight = teacher_magnitude * teacher_direction  # 合成同时含方向与幅值变化的教师权重。
training_inputs = torch.randn(24, input_size)  # 构造用于微型回归训练的输入批次。
training_targets = training_inputs @ teacher_weight.transpose(0, 1) + base_bias  # 用教师权重产生无噪声监督目标。
def mean_squared_error(predictions, targets):  # 手写均方误差以清楚展示标量训练目标。
    difference = predictions - targets  # 计算每个样本与特征位置的预测残差。
    return torch.mean(difference * difference)  # 对平方残差取平均得到可反传损失。
loss_before = mean_squared_error(student(training_inputs), training_targets)  # 计算参数更新前的回归损失。
loss_before.backward()  # 通过归一化、低秩矩阵和幅值执行第一次反向传播。
assert student.magnitude.grad is not None and torch.linalg.vector_norm(student.magnitude.grad) > 0.0  # 验证幅值参数第一步即可收到非零梯度。
assert student.adapter_b.grad is not None and torch.linalg.vector_norm(student.adapter_b.grad) > 0.0  # 验证零初始化矩阵 B 第一步即可收到非零梯度。
assert student.adapter_a.grad is not None and torch.count_nonzero(student.adapter_a.grad).item() == 0  # 验证 B 为零时矩阵 A 的首步梯度按链式法则为零。
assert all(parameter.grad is None or torch.isfinite(parameter.grad).all() for parameter in student.parameters())  # 验证所有已产生的首步梯度都是有限数值。
with torch.no_grad():  # 关闭更新操作的梯度记录以手写一次梯度下降。
    for parameter in student.parameters():  # 遍历 A、B 和幅值三个可训练参数。
        parameter -= 0.1 * parameter.grad  # 按固定学习率执行显式梯度下降更新。
        parameter.grad = None  # 清空首步梯度以免下一次反向传播累加。
loss_after_one_step = mean_squared_error(student(training_inputs), training_targets)  # 计算第一次手工更新后的损失。
loss_after_one_step.backward()  # 再次反向传播以检查更新 B 后 A 的梯度。
assert loss_after_one_step < loss_before  # 验证沿首步负梯度方向更新降低了训练损失。
assert student.adapter_a.grad is not None and torch.linalg.vector_norm(student.adapter_a.grad) > 0.0  # 验证 B 更新后梯度已经能流入矩阵 A。
assert torch.allclose(student.base_weight, frozen_weight_snapshot)  # 验证整个训练步骤没有修改冻结基座权重。

## 6. DoRA 怎么 merge：发布的是最终有效权重，不是简单把 BA 加进去

LoRA 的合并通常是 $W_{merge}=W_0+sBA$。DoRA 不能照搬这个表达式，因为它还必须执行逐向量归一化并乘上学习后的 magnitude：$W_{merge}=m\odot\operatorname{normalize}(W_0+sBA)$。漏掉归一化或 magnitude 都会让离线合并结果与训练时前向不一致。

可靠发布应生成一个普通线性层可直接加载的权重，同时在 manifest 中记录基座标识、矩阵布局、归一化轴、epsilon、dtype 和适配器版本。合并前后要用固定样本做数值等价检查；若之后量化，应先定义“先 merge 再量化”还是支持运行时适配器，不能把两个不同顺序的误差混为一谈。

In [ ]:
def build_merged_artifact(module, base_identifier):  # 把训练中的 DoRA 模块转换成普通线性层发布制品。
    merged_weight = module.effective_weight().detach().clone()  # 固化归一化和幅值缩放后的最终有效权重。
    merged_bias = module.base_bias.detach().clone()  # 复制与最终权重配套的冻结偏置。
    metadata = {"base_identifier": base_identifier, "layout": "out_in", "normalization_axis": 1, "epsilon": module.minimum, "dtype": str(merged_weight.dtype)}  # 记录重建与审计所需的关键元数据。
    return {"weight": merged_weight, "bias": merged_bias, "metadata": metadata}  # 返回不再依赖 A、B、magnitude 的独立制品。
merged_artifact = build_merged_artifact(student, "toy-base-v1")  # 将完成一次更新的学生层合并成普通权重制品。
live_adapter_outputs = student(sample_inputs).detach()  # 计算保留 DoRA 结构时的在线适配器输出。
merged_outputs = sample_inputs @ merged_artifact["weight"].transpose(0, 1) + merged_artifact["bias"]  # 使用发布权重直接执行普通线性前向。
artifact_keys = set(merged_artifact.keys())  # 收集合并制品的顶层字段以验证边界。
assert artifact_keys == {"weight", "bias", "metadata"}  # 验证独立发布制品不再携带训练模块对象。
assert torch.allclose(live_adapter_outputs, merged_outputs, atol=1e-6)  # 验证合并前后对同一输入的输出数值等价。
assert merged_artifact["weight"].shape == base_weight.shape  # 验证合并权重可直接替换原线性层权重。
assert merged_artifact["metadata"]["normalization_axis"] == 1  # 验证制品明确记录本实现按输入维归一化。
assert "adapter_a" not in merged_artifact and "magnitude" not in merged_artifact  # 验证普通部署制品不要求运行时理解 DoRA 参数名。

## 7. 归一化稳定性与梯度几何：FP32 范数、epsilon 和切向梯度

混合精度训练中，方向候选可能是 BF16/FP16，但范数最好在 FP32 中累加；极小值平方可能下溢，极大值平方也可能溢出。`epsilon` 是数值保护，不应大到显著改变正常权重。监控项至少包括最小/最大行范数、非有限值数量、接近零的行数以及 magnitude 的分布漂移。

对 $u=v/||v||$ 求导时，沿 $v$ 自身的径向分量会被投影掉，因此直接作用于单位方向的损失，其 $\partial L/\partial v$ 理论上近似垂直于 $v$。这正是方向更新和幅值更新能够分工的几何原因之一。浮点误差下不应断言点积严格为零，而应使用尺度相关容差。

In [ ]:
half_precision_probe = torch.tensor([[1e-4, -2e-4, 3e-4, 0.0, 1e-4, -1e-4], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], dtype=torch.float16)  # 构造极小半精度行和退化零行。
half_precision_unit = stable_row_normalize(half_precision_probe, epsilon)  # 使用内部 FP32 范数对半精度权重归一化。
geometry_probe = (torch.randn(output_size, input_size) + 0.5).requires_grad_()  # 构造非零方向候选并开启梯度记录。
geometry_target = torch.randn(output_size, input_size)  # 构造作用于单位方向的任意线性目标。
geometry_unit = stable_row_normalize(geometry_probe, epsilon)  # 对候选行归一化以形成方向变量。
geometry_loss = torch.sum(geometry_unit * geometry_target)  # 构造只依赖单位方向的标量损失。
geometry_loss.backward()  # 计算归一化操作对原方向候选的梯度。
radial_gradient_component = torch.sum(geometry_probe.detach() * geometry_probe.grad, dim=1)  # 计算每行梯度在原候选径向上的未归一化投影。
gradient_scale = torch.linalg.vector_norm(geometry_probe.detach(), dim=1) * torch.linalg.vector_norm(geometry_probe.grad, dim=1)  # 计算用于相对容差的向量范数乘积。
assert torch.isfinite(half_precision_unit).all()  # 验证半精度极小值与零行都不会产生非有限数值。
assert torch.count_nonzero(half_precision_unit[1]).item() == 0  # 验证退化零行在归一化后仍保持为零。
assert torch.allclose(stable_row_norm(half_precision_unit[:1]).float(), torch.ones(1, 1), atol=2e-3)  # 验证非零半精度行近似归一化为单位长度。
assert torch.all(torch.abs(radial_gradient_component) <= 1e-5 * gradient_scale.clamp_min(1.0))  # 验证方向归一化梯度的径向分量在浮点容差内接近零。
assert torch.isfinite(geometry_probe.grad).all()  # 验证归一化反向传播得到的梯度均为有限数值。

## 8. 参数量、训练制品与上线审计

LoRA 每层训练参数量为 $r(d_{in}+d_{out})$；DoRA 额外增加 $d_{out}$ 个 magnitude 参数，在大矩阵上通常仍远少于全量权重。训练 checkpoint 应保存 A、B、magnitude 以及足以唯一定位基座和超参数的 manifest；只保存三组张量却不记录基座版本，会使同名适配器在不同 checkpoint 上静默地产生错误权重。

上线前至少检查：所有参数有限；有效权重行范数与 magnitude 一致；基座哈希匹配；实时前向与 merged 前向在容差内一致；训练 dtype、合并 dtype 和最终量化策略明确；卸载适配器能恢复基座。若要支持多个适配器动态切换，应缓存或分批计算范数，并评估额外延迟和显存，而不是默认 DoRA 与 LoRA 的服务成本完全相同。

In [ ]:
def audit_dora_module(module, probe_inputs, tolerance=1e-5):  # 对训练模块执行发布前的关键一致性审计。
    effective_weight = module.effective_weight().detach()  # 固化当前有效权重供多项检查复用。
    effective_norm = stable_row_norm(effective_weight, module.minimum)  # 计算最终权重每个输出行的实际范数。
    live_outputs = module(probe_inputs).detach()  # 计算保留适配器结构时的参考输出。
    merged_outputs = probe_inputs @ effective_weight.transpose(0, 1) + module.base_bias  # 计算合并为普通线性权重后的输出。
    return {"parameters_finite": all(torch.isfinite(parameter).all().item() for parameter in module.parameters()), "norm_matches_magnitude": torch.allclose(effective_norm, module.magnitude.detach().abs(), atol=tolerance), "merge_max_error": torch.max(torch.abs(live_outputs - merged_outputs)).item()}  # 返回有限性、范数和合并误差三项审计结果。
audit_report = audit_dora_module(student, sample_inputs)  # 对完成梯度实验的学生模块执行发布审计。
lora_trainable_count = rank * input_size + output_size * rank  # 计算同尺寸 LoRA 的 A 与 B 参数总数。
dora_trainable_count = lora_trainable_count + output_size  # 加上每个输出行一个 magnitude 得到 DoRA 参数量。
full_weight_count = input_size * output_size  # 计算当前微型层全量训练原始权重所需的参数数量。
large_input_size = 4096  # 设置一个更接近大模型投影层的示例输入维度。
large_output_size = 4096  # 设置一个更接近大模型投影层的示例输出维度。
large_rank = 8  # 设置大模型参数高效微调常见的小秩示例。
large_dora_count = large_rank * (large_input_size + large_output_size) + large_output_size  # 计算大矩阵示例中的 DoRA 可训练参数量。
large_full_count = large_input_size * large_output_size  # 计算大矩阵示例中的全量权重参数量。
assert audit_report["parameters_finite"]  # 验证所有 DoRA 可训练参数都没有非有限值。
assert audit_report["norm_matches_magnitude"]  # 验证最终权重行范数与显式幅值参数一致。
assert audit_report["merge_max_error"] < 1e-6  # 验证实时适配器前向与合并权重前向等价。
assert dora_trainable_count - lora_trainable_count == output_size  # 验证 DoRA 相比 LoRA 只增加逐输出幅值参数。
assert dora_trainable_count == full_weight_count  # 验证微型高相对秩配置恰好不节省参数并暴露低秩方法的边界。
assert large_dora_count < large_full_count  # 验证在真实大矩阵与小秩组合下 DoRA 参数量远小于全量训练。
assert set(audit_report.keys()) == {"parameters_finite", "norm_matches_magnitude", "merge_max_error"}  # 验证审计报告包含约定的全部核心字段。

## 9. 面试总结：从公式回答到工程落地

完整回答可以按五步组织：第一，LoRA 冻结 $W_0$ 并用 $BA$ 表示低秩更新；第二，DoRA 把 $W_0+sBA$ 当作方向候选，归一化后再乘独立可训练 magnitude；第三，初始化 `B=0` 且 `m=||W0||`，保证起点与基座等价；第四，方向归一化梯度主要处于切空间，magnitude 负责径向变化，同时范数要用 FP32 和 epsilon 做稳定保护；第五，merge 必须固化完整公式，而不是简单执行 LoRA 式加法，并用前向等价、基座身份和 manifest 做发布校验。

还应主动说明边界：DoRA 不保证所有任务都优于 LoRA，也没有消除秩、目标层、学习率和数据质量等超参数；本实验只验证公式、梯度路径与制品契约，并非论文规模效果复现。真实训练要补齐分布式切分、混合精度、激活检查点、恢复训练、量化顺序、多个适配器并存及端到端评测。能把这些失败模式说清楚，才是“理解 DoRA”，而不只是背诵幅值与方向两个名词。